## algorithm
initial data - EI BO analyse - return x - for loop(call comsol - input x - output y - EI BO analyse - return x ) - end loop(if abs(old x - new x) < 0.000001)

In [22]:
# make a list of initial data
import glob
import pandas as pd
import os
import torch

def trainX(model):
    df = pd.read_table(model, encoding='cp1252', header=None)
    # para = df[0].str.contains('Parameterkombination')
    para = df[0][0].split('{')
    para = [i.replace("}", "") for i in para]
    para = [j.replace("'", "") for j in para]
    para = [j.replace(" ", "") for j in para]
    para_ = para[1].split(',')
    parameter = {
        k:float(v)
        for k,v in(
            item.split(':',1) for item in para_
        )
    }
    df_para = pd.DataFrame([parameter])

    m1_w = df_para['M1_W'].item()
    m1_h = df_para['M1_H'].item()
    m2_h = df_para['M2_H'].item()
    m2_w = df_para['M2_W'].item()
    offset = df_para['offset'].item()
    return [m1_h, m1_w, m2_h, m2_w, offset]

def trainY(model):
    df = pd.read_table(model, encoding="cp1252", sep=r"\s+",
    comment="#",
    header=None,    
    names=["Zeit", "Fx"])
    return df['Fx'].mean()

def train_set(DataFolder):
    exp_files = glob.glob(os.path.join(DataFolder, "*.txt"))
    train_x_list = []
    train_y_list = []
    train_con_list = []

    for files in exp_files:
        # print(files)
        x = trainX(files)
        y = trainY(files)
    
        train_x_list.append(x)
        train_y_list.append(y)
   
    
    train_x = torch.tensor(train_x_list, dtype=torch.float64)
    train_y = torch.tensor(train_y_list, dtype=torch.float64)
    train_con = torch.tensor(train_con_list,dtype = torch.float64)
    # print(train_x)
    return train_x, train_y

In [23]:
# Expected Improvement Bayesian Optimization
import os
import torch
import warnings
from typing import Optional

from botorch import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.acquisition.objective import GenericMCObjective
from botorch.exceptions import BadInitialCandidatesWarning
from botorch.models.transforms.input import Normalize

from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.acquisition import qLogExpectedImprovement, qLogNoisyExpectedImprovement

from trainset import trainX, trainY, train_set

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

# constrains
def constraint_ftn(X):
    return X[:,1]+X[:,3]-15 # if the zero point is 3mm to 12mm, moving the boundary btw magnets

# evaluate constrains
def weighted_obj(X, exact_obj_):
    """feasible 1; not feasible 0"""
    value = exact_obj_.unsqueeze(-1) if exact_obj_.ndim==1 else exact_obj_
    return value*(constraint_ftn(X) <= 0).type_as(X)

# constrained objective wrapper
def obj_callable(Z: torch.Tensor, X: Optional[torch.Tensor] = None):
    return Z[..., 0]
def constraint_callable(Z):
    return Z[..., 1]

# build model
def generate_model(train_x, train_obj, train_con, train_yvar, state_dict = None):
    model_obj = SingleTaskGP(
        train_x,
        train_obj,
        train_yvar.expand_as(train_obj),
        input_transform=Normalize(d=train_x.shape[-1]),
        ).to(train_x)
    model_con = SingleTaskGP(
        train_x,
        train_con,
        train_yvar.expand_as(train_con),
    input_transform = Normalize(d=train_x.shape[-1],)
        ).to(train_x)
    model = ModelListGP(model_obj, model_con)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    if state_dict is not None:
        model.load_state_dict(state_dict)
    return mll, model

# acquisit function
def optimize_acqf_and_get_observation(acq_func, BATCH_SIZE, NUM_RESTARTS, RAW_SAMPLES, NOISE_SE):
    """Optimizes the acquistion function, and returns a new candidate and a noisy observation"""
    candidates, _ = optimize_acqf(
        acq_function = acq_func,
        bounds=torch.tensor([[0.1, 3.0, 0.1, 3.0, -3.0], [15.0, 9.0, 15.0, 9.0, 5.0]]),#check lower and upper bound!
        q = BATCH_SIZE, # one optimal return value
        num_restarts = NUM_RESTARTS,
        raw_samples = RAW_SAMPLES,
        options = {"batch_limit": 5, "maxiter": 200},
        )
        
    new_x = candidates.detach()
    exact_con = constraint_ftn(new_x).unsqueeze(-1)
    new_con = exact_con + NOISE_SE * torch.randn_like(exact_con)
    return new_x, new_con


def BayesianOptimization(train_x, train_y):
    BATCH_SIZE = 1
    NUM_RESTARTS = 10 if not SMOKE_TEST else 2
    RAW_SAMPLES = 512 if not SMOKE_TEST else 32
    MC_SAMPLES = 256 if not SMOKE_TEST else 32

    # generate noise    
    NOISE_SE = 0.25
    train_yvar = torch.tensor(NOISE_SE**2, device=device, dtype=dtype)

    # generate data
    exact_obj = train_y.unsqueeze(-1)
    train_obj = exact_obj + NOISE_SE*torch.randn_like(exact_obj)
    exact_con = constraint_ftn(train_x).unsqueeze(-1)
    train_con = exact_con + NOISE_SE*torch.randn_like(exact_con)
    # best_observed_value = train_obj.max().item() # qExpectedImprovement for acuisitfunction in the code, qNoisyExpectedImprovment for unreliable small data

    # call the objective
    objective = GenericMCObjective(objective= obj_callable)

    warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    verbose = False

    mll_nei, model_nei = generate_model(train_x, train_obj, train_con, train_yvar)


    fit_gpytorch_mll(mll_nei)

    qmc_sampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]))

    qLogNEI = qLogNoisyExpectedImprovement(
        model=model_nei,
        X_baseline=train_x,
        sampler = qmc_sampler,
        objective=objective,
        constraints=[constraint_callable],
        )
        
    new_x_nei, new_con_nei = optimize_acqf_and_get_observation(qLogNEI, BATCH_SIZE, NUM_RESTARTS, RAW_SAMPLES, NOISE_SE)

    train_x = torch.cat([train_x, new_x_nei])
   
    train_con = torch.cat([train_con, new_con_nei])

    # print('M1_H', 'M1_W', 'M2_H', 'M2_W', 'offset', sep=" ")
    # print(new_x_nei)
    return new_x_nei



In [24]:
from comsol_interface import run_one
from pathlib import Path
from config import ALLOWED_LEFT_BOUNDRY, ALLOWED_RIGHT_BOUNDRY, W_MIN, W_MAX, H_MIN, H_MAX, OFFSET_MIN, OFFSET_MAX, AUSDRUCK, DATENSATZ
from datetime import datetime
import numpy as np
import mph

def combo_label(c: dict) -> str:
    label = f"M1H{c['M1_H']}_M1W{c['M1_W']}_M2H{c['M2_H']}_M2W{c['M2_W']}_OFF{c['offset']}"
    return label.replace(".", "p")                  # p for pont, 1.6 => 1 point 6 => 1p6

RESULTS_DIR  = Path("results03")

def save_single_result(combo: dict, zeiten: np.ndarray, fx_werte: np.ndarray):

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    fname = RESULTS_DIR / f"{combo_label(combo)}.txt"
    header = (
        f"Parameterkombination: {combo}\n"
        f"Ausdruck: {AUSDRUCK}\n"
        f"Datensatz: {DATENSATZ}\n"
        f"Erstellt: {datetime.now().isoformat()}\n"
        f"{'Zeit [s]':>16}  {'Fx [N]':>16}"
    )

    np.savetxt(
        fname,
        np.column_stack([zeiten, fx_werte]),
        fmt="%16.6e",
        header=header
    )

# Feste Parameter
FIXED_PARAMS = {
    "cooling_gap":    "4[mm]",                  # from 3 to 5. but it would be better to have an optimization for 3 mm, one for 4 mm and one for 5 mm. depending on what cooling gap ist achievable you can chose your magnet arrangement.
    "M_length":       "30[mm]",                 # 30 mm are realistic for measurement but can deviat
    "verschiebung":   "1",                      # currently not used                      
    "Ec":             "1e-4[V/m]",              # material parameter
}

def set_parameters(model, combo: dict):
    """Übergabe an COMSOL :)"""
    params = {
        **FIXED_PARAMS,
        "M1_H"   : f"{combo['M1_H']}[mm]",
        "M1_W"   : f"{combo['M1_W']}[mm]",
        "M2_H"   : f"{combo['M2_H']}[mm]",
        "M2_W"   : f"{combo['M2_W']}[mm]",
        "offset" : f"{combo['offset']}[mm]"
    }
    for name, value in params.items():
        model.parameter(name, value)

def run_one(model, combo: dict) -> tuple[np.ndarray, np.ndarray]:

    set_parameters(model, combo)
    model.solve()

    _, zeiten = model.inner(DATENSATZ)
    fx_werte = model.evaluate(AUSDRUCK, dataset=DATENSATZ)

    zeiten   = np.array(zeiten,   dtype=float)
    fx_werte = np.array(fx_werte, dtype=float)

    #---- Datei speichern -------------------------------------------------------
    fname = RESULTS_DIR / f"{combo_label(combo)}.txt"
    header = (
        f"Parameterkombination: {combo}\n"
        f"Ausdruck: {AUSDRUCK}\n"
        f"Datensatz: {DATENSATZ}\n"
        f"Erstellt: {datetime.now().isoformat()}\n"
        f"{'Zeit [s]':>16}  {'Fx [N]':>16}"
    )
    np.savetxt(fname, np.column_stack([zeiten, fx_werte]),
               fmt="%16.6e", header=header)
    #---------------------------------------------------------------------------

    return zeiten, fx_werte

# call the comsol
def comsol(input):
    print("Start Comsol Simulation")

    # tensor unpacking
    m1h, m1w, m2h, m2w, offset = input[0].float().tolist() # Aufgabe : check the arbitary values

    # define combo
    combo = {
            "M1_H":   round(m1h,    1),
            "M1_W":   round(m1w,    1),
            "M2_H":   round(m2h,    1),
            "M2_W":   round(m2w,    1),
            "offset": round(offset, 1),
        }
    try:
        client = mph.start()    # its important to have this on a "highl-level" because the client needs time to start
        model = client.load("3D_H-phi_PM-Pair_N38_ld1_z4_V3.mph")
        time, fx_val = run_one(model, combo)
        save_single_result(combo, time, fx_val)
        print(combo, f" Zeit: {time[-1]} ", f"Fx_Ende: {fx_val[-1]} [N]")
        return float(fx_val[-1]) # why fx_val[-1]? instead of np.mean(fx_val?) dependency train_set('results03')
    except Exception as e:
        print(f"Fehler bei Kombination {combo}: {e}")
        return -1e9
    return 


In [26]:

train_x, exact_obj = train_set('results03')

print(exact_obj)
for i in range(10):
    
    new_x = BayesianOptimization(train_x, exact_obj)
    # data type new_x is tensor(1,5)..1batch each batch has 5 values

    # run the COMSOL
    new_obj = comsol(new_x)
    
    train_x = torch.cat([train_x,new_x],dim=0)
    max = torch.max(exact_obj, dim=0)[0].item()#actually exact_obj should be maxmum check it, whether BO works well

    if new_obj - max < 1e-9:
        print("the maximum Fx is", max, "if parameters are ", new_x)
        break

    exact_obj = torch.cat([exact_obj,torch.tensor([new_obj])],dim=0)
    print("Bayesian Optimization iteration = ", i+1)

tensor([0.0030, 0.0048, 0.0058, 0.0059, 0.0055, 0.0059, 0.0046, 0.0024, 0.0032,
        0.0045, 0.0049, 0.0047, 0.0063, 0.0059, 0.0020, 0.0055, 0.0034, 0.0043],
       dtype=torch.float64)


Could not find a supported Comsol installation.


Start Comsol Simulation
Fehler bei Kombination {'M1_H': 13.0, 'M1_W': 5.5, 'M2_H': 5.2, 'M2_W': 3.0, 'offset': 3.0}: Could not find a supported Comsol installation.
the maximum Fx is 0.006281012307692429 if parameters are  tensor([[13.0426,  5.5292,  5.2133,  3.0000,  2.9970]])
